# 02 — EDA: Sales Analysis

## Purpose
EDA stands for **Exploratory Data Analysis** — it means studying the data to understand
patterns, distributions, and relationships before building any model.

This notebook answers the **core business questions**:
- Which branches sell the most?
- How do dine-in vs takeout compare?
- What are the best-selling products overall?
- What are the top products at each branch?
- Which products generate the most revenue?

## Input
`data/intermediate/datanomodifier.csv`

## Run order
Run after `01_data_cleaning.ipynb`.

In [ ]:
import os

# ─── PATH CONFIGURATION ───────────────────────────────────────────────
# Option A — After cloning the repo (default, USE_GITHUB = False)
#   git clone https://github.com/DiegoLarrieta/PanemReto
#   cd PanemReto/notebooks
#   jupyter notebook
#   Paths resolve automatically — no changes needed.
#
# Option B — Read directly from GitHub, e.g. Google Colab (USE_GITHUB = True)
#   Works for notebooks 07 (processed branch CSVs are on GitHub).
#   Notebooks 00-06 need the intermediate files which are NOT in the repo
#   (too large). Run 00_data_pipeline.ipynb locally first to generate them.
# ──────────────────────────────────────────────────────────────────────────

USE_GITHUB   = False
GITHUB_BASE  = "https://media.githubusercontent.com/media/DiegoLarrieta/PanemReto/main"

if USE_GITHUB:
    PROCESSED_DIR    = f"{GITHUB_BASE}/data/processed"
    WEATHER_DIR      = f"{GITHUB_BASE}/data/weather"
    RAW_DIR          = f"{GITHUB_BASE}/data/raw/Complete Data"
    INTERMEDIATE_DIR = None  # not in repo — generate locally with 00_data_pipeline.ipynb
else:
    PROCESSED_DIR    = PROCESSED_DIR
    WEATHER_DIR      = WEATHER_DIR
    RAW_DIR          = RAW_DIR
    INTERMEDIATE_DIR = INTERMEDIATE_DIR


## Setup — Load and clean data

Each analysis notebook starts with the same load + clean block so it can run independently.

## Analysis 1 — Total sales by branch (quantity)

Which branch sells the most units? This gives us the relative size of each location
and helps us contextualize branch-level comparisons in later analyses.

In [ ]:
top_sucursal_qty = (
    base.groupby("sucursal", as_index=False)["quantity"]
        .sum()
        .rename(columns={"quantity": "qty_sold"})
        .sort_values("qty_sold", ascending=False)
        .reset_index(drop=True)
)

top_sucursal_qty["pct"] = (top_sucursal_qty["qty_sold"] / top_sucursal_qty["qty_sold"].sum() * 100).round(1)
top_sucursal_qty

## Analysis 2 — Sales by channel: Restaurant vs Para Llevar

**Para llevar** = takeout. **Restaurant** = dine-in.

Understanding the channel mix matters for forecasting because takeout orders tend to be
more consistent and predictable, while dine-in can spike with events or bad weather.

In [ ]:
# Pivot table: rows = branch, columns = order_type, values = quantity
channel_qty = (
    base.pivot_table(
        index="sucursal",
        columns="order_type",
        values="quantity",
        aggfunc="sum",
        fill_value=0
    )
)

channel_qty["TOTAL"] = channel_qty.sum(axis=1)
channel_qty = channel_qty.sort_values("TOTAL", ascending=False)

print("=== Units sold by channel ===")
display(channel_qty)

# Percentage mix: what % of each branch's sales is takeout vs dine-in
channel_pct = (
    channel_qty.drop(columns="TOTAL")
    .div(channel_qty["TOTAL"], axis=0)
    .mul(100).round(1)
)

print("\n=== Channel mix % per branch ===")
display(channel_pct)

## Analysis 3 — Top items overall (by units sold)

Across all 7 branches, which products are sold the most?
This list gives us a global view of Panem's most popular products.

In [ ]:
top_items_overall = (
    base.groupby("item", as_index=False)["quantity"]
        .sum()
        .rename(columns={"quantity": "qty_sold"})
        .sort_values("qty_sold", ascending=False)
        .reset_index(drop=True)
)
top_items_overall.index = top_items_overall.index + 1
top_items_overall.head(20)

## Analysis 4 — Top 20 items per branch (by units sold)

Each branch has its own customer base and product mix. A product that ranks #1 across
all branches may not even appear in the top 20 at a specific location.

This analysis shows us the top 20 sellers **per branch** — the foundation for identifying
the top 5 products we need to forecast.

In [ ]:
TOP_N = 20

top_per_branch = (
    base.groupby(["sucursal", "item"], as_index=False)["quantity"]
        .sum()
        .rename(columns={"quantity": "qty_sold"})
        .sort_values(["sucursal", "qty_sold"], ascending=[True, False])
        .groupby("sucursal", as_index=False)
        .head(TOP_N)
)

for branch in sorted(top_per_branch["sucursal"].dropna().unique()):
    print(f"\n{'='*50}")
    print(f"  {branch}")
    print(f"{'='*50}")
    table = (
        top_per_branch[top_per_branch["sucursal"] == branch][["item", "qty_sold"]]
        .reset_index(drop=True)
    )
    table.index = table.index + 1
    display(table)

## Analysis 5 — Top 20 items per branch (by revenue)

Units sold and revenue don't always agree. A cheap product sold in high volume
may rank below a premium item sold less often when measured in pesos.

Comparing both rankings helps us understand whether top-5 predictions should
optimize for volume or revenue — or both.

In [ ]:
base["total_ticket"] = pd.to_numeric(base["total_ticket"], errors="coerce").fillna(0)

top_revenue_per_branch = (
    base.groupby(["sucursal", "item"], as_index=False)["total_ticket"]
        .sum()
        .rename(columns={"total_ticket": "revenue_mxn"})
        .sort_values(["sucursal", "revenue_mxn"], ascending=[True, False])
        .groupby("sucursal", as_index=False)
        .head(TOP_N)
)

for branch in sorted(top_revenue_per_branch["sucursal"].dropna().unique()):
    print(f"\n{'='*50}")
    print(f"  {branch}")
    print(f"{'='*50}")
    table = (
        top_revenue_per_branch[top_revenue_per_branch["sucursal"] == branch][["item", "revenue_mxn"]]
        .reset_index(drop=True)
    )
    table.index = table.index + 1
    display(table)

## Summary

Key takeaways from this notebook:
- Branch volume varies significantly (check the `pct` column in Analysis 1)
- Each branch has a distinct product mix — a global model would miss local patterns
- Volume and revenue rankings differ per branch — review both before finalizing top-5 lists

**Next step:** `03_eda_weather_temporal.ipynb` — understand how temperature, day of week,
and time of day affect what customers buy.